# Personal Knowledge Assistant
**Agentic AI Course · Final Project**

A 3-agent CrewAI crew that answers questions from *my own documents*:
1. **Researcher** - searches my documents (RAG)
2. **Verifier** - checks what the Researcher found against the live web
3. **Writer** - synthesizes everything into one clear answer (no tool)

The crew also has memory, so it can recall earlier questions across runs.

## Step 1 - Environment setup

## Step 2 - Upload and prepare your documents
Upload your own `.txt` and/or `.pdf` file(s). Re-run this cell every time the Colab session restarts.

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

In [ ]:
from pypdf import PdfReader

document_text = ""
for filename in uploaded.keys():
    if filename.lower().endswith(".txt"):
        with open(filename, "r", encoding="utf-8") as f:
            document_text += f.read() + "\n"
    elif filename.lower().endswith(".pdf"):
        reader = PdfReader(filename)
        for page in reader.pages:
            document_text += (page.extract_text() or "") + "\n"
    else:
        print(f"Skipping unsupported file: {filename}")

print(f"Total characters: {len(document_text)}")
print(document_text[:300])  # sanity check

## Step 3 - Chunk and embed

In [ ]:
def chunk_text(text, chunk_size=500, overlap=50):
    """Split text into overlapping chunks."""
    chunks = []
    step = chunk_size - overlap
    start = 0
    while start < len(text):
        chunk = text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        start += step
    return chunks

chunks = chunk_text(document_text)
print(f"Created {len(chunks)} chunks")

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = embedder.encode(chunks)
print("Embeddings shape:", chunk_embeddings.shape)

## Step 4 - Test retrieval on its own

In [ ]:
import numpy as np

def retrieve(query, top_k=3):
    """Embed the query and return the top_k most similar chunks (cosine similarity)."""
    q = embedder.encode([query])[0]
    sims = chunk_embeddings @ q / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(q) + 1e-10
    )
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [chunks[i] for i in top_idx]

# TODO: replace with a real question about YOUR document
results = retrieve("a real question about your document")
for r in results:
    print(r, "\n---")

> **COMMIT 1** - "RAG pipeline working - documents chunked, embedded, and retrievable"

## Step 5 - Document Search tool + Researcher

In [ ]:
@tool("Document Search")
def document_search(query: str) -> str:
    """Searches the student's own uploaded documents for relevant passages."""
    results = retrieve(query)
    return str(results)

researcher = Agent(
    role="Personal Notes Researcher",
    goal="Find the passages in the student's own uploaded documents that are most relevant to the question: {question}",
    backstory=(
        "You are a careful research assistant who knows only what is written in the "
        "student's own notes and documents. You never use the open web and never invent "
        "facts - you only report what the documents actually say."
    ),
    tools=[document_search],
    llm=llm,
    verbose=True,
)

## Step 6 - Web Search tool + Verifier

In [ ]:
@tool("Web Search")
def web_search(query: str) -> str:
    """Searches the web for current information on a given query."""
    from ddgs import DDGS
    results = DDGS().text(query, max_results=3)
    return str(results)

verifier = Agent(
    role="Fact Verifier",
    goal="Check whether claims found in the student's notes are still correct according to the live web.",
    backstory=(
        "You are a skeptical fact-checker. Student notes can be outdated or simply wrong, "
        "so you never just agree with the Researcher - you always search the web and "
        "compare what you find with what the notes claim."
    ),
    tools=[web_search],
    llm=llm,
    verbose=True,
)

> **COMMIT 2** - "Researcher and Verifier agents working against your documents and the web"

## Step 7 - Writer + full crew

In [ ]:
writer = Agent(
    role="Answer Writer",
    goal="Write one clear, accurate answer to the student's question using the retrieved notes and the verification result.",
    backstory=(
        "You are a clear, honest writer. You combine what the student's notes say with what "
        "the verifier found, and you always point out anything in the notes that is outdated "
        "or wrong. You do not need any tools - your job is synthesis only."
    ),
    llm=llm,  # deliberately no tools=[...]
    verbose=True,
)

research_task = Task(
    description=(
        "The student asked: {question}\n\n"
        "Use the Document Search tool to find the most relevant passages in the student's own "
        "documents. You may search more than once with different wording. Return the relevant "
        "passages and list the specific factual claims they make."
    ),
    agent=researcher,
    expected_output="The relevant passages from the documents, plus a short bullet list of the factual claims they contain.",
)

verify_task = Task(
    description=(
        "Take the specific factual claims from the Researcher's findings for the question: {question}\n\n"
        "You MUST call the Web Search tool at least once for the key claims (do not skip this). "
        "For each claim, say whether the web CONFIRMS it, CONTRADICTS it, or whether it is "
        "UNCLEAR, and explain briefly using what you found."
    ),
    agent=verifier,
    context=[research_task],
    expected_output="A list of claims, each marked CONFIRMED / CONTRADICTED / UNCLEAR with a one-line justification from the web results.",
)

write_task = Task(
    description=(
        "Write the final answer to the student's question: {question}\n\n"
        "Base it on the retrieved notes and the verification result. If any claim in the notes "
        "was contradicted or unclear, say so clearly. Keep it concise and easy to read."
    ),
    agent=writer,
    context=[research_task, verify_task],
    expected_output="A clear final answer (a few short paragraphs at most) that notes any outdated or incorrect information.",
)

crew = Crew(
    agents=[researcher, verifier, writer],
    tasks=[research_task, verify_task, write_task],
    process=Process.sequential,
)

# TODO: replace with a real question about YOUR document
result = crew.kickoff(inputs={"question": "your real question here"})
print(result.raw)

> **COMMIT 3** - "Full 3-agent crew producing answers end to end"

## Step 8 - Enable memory
CrewAI memory defaults to OpenAI embeddings, so we pass an explicit HuggingFace embedder (no OpenAI key needed).

In [ ]:
crew = Crew(
    agents=[researcher, verifier, writer],
    tasks=[research_task, verify_task, write_task],
    process=Process.sequential,
    memory=True,
    embedder={
        "provider": "huggingface",
        "config": {"model": "sentence-transformers/all-MiniLM-L6-v2"},
    },
)

Ask two **related** questions in separate `kickoff()` calls and compare the outputs.

In [ ]:
# TODO: replace with two related questions about YOUR document
result1 = crew.kickoff(inputs={"question": "First question about your document"})
print(result1.raw)

In [ ]:
result2 = crew.kickoff(inputs={"question": "A follow-up question related to the first"})
print(result2.raw)

> **COMMIT 4** - "Memory enabled - two related runs show the crew recalling context"

**Memory observation (fill in after testing):**
Did the second answer show any sign of recalling the first? Write what you actually saw here.

> **COMMIT 5** - "README and sample Q&A added - final submission"